# DML and the Basic SELECT Pipeline using SQLite

This notebook covers SQL basics for Data Engineering without setting up a local database server.

Tools used:
- Google Colab
- Python `sqlite3`
- Pandas for displaying SQL query results

Topics covered:
- INSERT INTO: single insert and bulk insert
- UPDATE and DELETE
- Why missing `WHERE` is dangerous
- SELECT specific columns vs `SELECT *`
- DISTINCT
- Aliases using `AS`
- WHERE filters with `AND`, `OR`, `NOT`
- IN, BETWEEN, LIKE, IS NULL
- SQL execution order
- Interview questions


## 1. Setup: SQLite in Memory

SQLite is built into Python, so no external database setup is required.

The database will exist only while this notebook session is running.


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

print("SQLite database created in memory")


SQLite database created in memory


In [ ]:
cursor.execute("SELECT * FROM customers;")

## 2. Helper Functions

`run_sql()` is used for SELECT queries and returns a Pandas DataFrame.

`execute_sql()` is used for INSERT, UPDATE, DELETE, and CREATE statements.


In [ ]:
def run_sql(query):
    return pd.read_sql_query(query, conn)


def execute_sql(query, params=None):
    if params is None:
        cursor.execute(query)
    else:
        cursor.execute(query, params)
    conn.commit()


def execute_many(query, records):
    cursor.executemany(query, records)
    conn.commit()


## 3. Create Empty Tables

These tables simulate a small e-commerce database.

Tables:
- `customers`
- `orders`


In [ ]:
cursor.executescript("""

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    city TEXT,
    segment TEXT,
    email TEXT
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    amount REAL,
    status TEXT,
    coupon_code TEXT
);
""")

conn.commit()

print("Tables created")


Tables created


# Data Manipulation Language: DML

DML is used to change data inside tables.

Common DML commands:
- `INSERT INTO`
- `UPDATE`
- `DELETE`


## 4. INSERT INTO: Single Insert

A single insert adds one row into a table.


In [ ]:
execute_sql("""
INSERT INTO customers (customer_id, customer_name, city, segment, email)
VALUES (501, 'Riya', 'Delhi', 'Premium', 'riya@example.com')
""")

run_sql("SELECT * FROM customers")


,customer_id,customer_name,city,segment,email
0,501,Riya,Delhi,Premium,riya@example.com


## 5. INSERT INTO: Bulk Insert

Bulk insert adds multiple rows efficiently.

In Python, `executemany()` is useful for inserting many records.


In [ ]:
customer_records = [
    (502, 'Aarav', 'Mumbai', 'Standard', 'aarav@example.com'),
    (503, 'Kabir', 'Bangalore', 'Standard', None),
    (504, 'Meera', 'Pune', 'Premium', 'meera@example.com'),
    (505, 'Anz', 'Chennai', 'Standard', 'anz@example.com'),
    (506, 'AnushkaZ', 'Delhi', 'Premium', 'anushkaz@example.com'),
    (507, 'Aliz', None, 'Standard', None)
]

execute_many("""
INSERT INTO customers (customer_id, customer_name, city, segment, email)
VALUES (?, ?, ?, ?, ?)
""", customer_records)

run_sql("SELECT * FROM customers")


,customer_id,customer_name,city,segment,email
0,501,Riya,Delhi,Premium,riya@example.com
1,502,Aarav,Mumbai,Standard,aarav@example.com
2,503,Kabir,Bangalore,Standard,None
3,504,Meera,Pune,Premium,meera@example.com
4,505,Anz,Chennai,Standard,anz@example.com
5,506,AnushkaZ,Delhi,Premium,anushkaz@example.com
6,507,Aliz,None,Standard,None


## 6. Insert Orders

The orders table intentionally contains:
- repeated statuses
- missing coupon codes
- one invalid customer ID
- one missing amount

These issues will help with filtering and data-quality examples.


In [ ]:
order_records = [
    (1001, 501, '2026-01-01', 2500.0, 'completed', 'NEW10'),
    (1002, 502, '2026-01-02', 1800.0, 'completed', None),
    (1003, 501, '2026-01-03', 3200.0, 'pending', 'SAVE20'),
    (1004, 503, '2026-01-04', 900.0, 'completed', None),
    (1005, 999, '2026-01-05', 700.0, 'completed', None),
    (1006, 504, '2026-01-06', 5000.0, 'cancelled', 'FEST'),
    (1007, 505, '2026-01-07', None, 'completed', None),
    (1008, 507, '2026-01-08', 1200.0, 'completed', 'A1')
]

execute_many("""
INSERT INTO orders (order_id, customer_id, order_date, amount, status, coupon_code)
VALUES (?, ?, ?, ?, ?, ?)
""", order_records)

run_sql("SELECT * FROM orders")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1001,501,2026-01-01,2500.0,completed,NEW10
1,1002,502,2026-01-02,1800.0,completed,None
2,1003,501,2026-01-03,3200.0,pending,SAVE20
3,1004,503,2026-01-04,900.0,completed,None
4,1005,999,2026-01-05,700.0,completed,None
5,1006,504,2026-01-06,5000.0,cancelled,FEST
6,1007,505,2026-01-07,NaN,completed,None
7,1008,507,2026-01-08,1200.0,completed,A1


## 7. UPDATE with WHERE

`UPDATE` changes existing records.

Always use a `WHERE` clause unless the intention is to update every row.


In [ ]:
execute_sql("""
UPDATE orders
SET status = 'completed'
WHERE order_id = 1003
""")

run_sql("SELECT * FROM orders WHERE order_id = 1003")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1003,501,2026-01-03,3200.0,completed,SAVE20


## 8. Danger of UPDATE Without WHERE

The following example uses a demo table so the original data is protected.

Without `WHERE`, every row is updated.


In [ ]:
cursor.executescript("""
DROP TABLE IF EXISTS orders_update_demo;
CREATE TABLE orders_update_demo AS
SELECT * FROM orders;
""")
conn.commit()

execute_sql("""
UPDATE orders_update_demo
SET status = 'archived'
""")

run_sql("""
SELECT status, COUNT(*) AS record_count
FROM orders_update_demo
GROUP BY status
""")


,status,record_count
0,archived,8


## 9. DELETE with WHERE

`DELETE` removes rows.

Always use a `WHERE` clause unless the intention is to delete every row.


In [ ]:
cursor.executescript("""
DROP TABLE IF EXISTS orders_delete_demo;
CREATE TABLE orders_delete_demo AS
SELECT * FROM orders;
""")
conn.commit()

execute_sql("""
DELETE FROM orders_delete_demo
WHERE status = 'cancelled'
""")

run_sql("SELECT * FROM orders_delete_demo")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1001,501,2026-01-01,2500.0,completed,NEW10
1,1002,502,2026-01-02,1800.0,completed,None
2,1003,501,2026-01-03,3200.0,completed,SAVE20
3,1004,503,2026-01-04,900.0,completed,None
4,1005,999,2026-01-05,700.0,completed,None
5,1007,505,2026-01-07,NaN,completed,None
6,1008,507,2026-01-08,1200.0,completed,A1


## 10. Danger of DELETE Without WHERE

Without `WHERE`, all rows are deleted.

The example below uses a demo table.


In [ ]:
cursor.executescript("""
DROP TABLE IF EXISTS orders_delete_danger_demo;
CREATE TABLE orders_delete_danger_demo AS
SELECT * FROM orders;
""")
conn.commit()

execute_sql("DELETE FROM orders_delete_danger_demo")

run_sql("SELECT COUNT(*) AS remaining_rows FROM orders_delete_danger_demo")


,remaining_rows
0,0


# The SELECT Statement

`SELECT` is used to retrieve data from tables.


## 11. SELECT *

`SELECT *` returns all columns.

It is useful for quick exploration but not ideal for production queries.


In [ ]:
run_sql("""
SELECT *
FROM orders
""")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1001,501,2026-01-01,2500.0,completed,NEW10
1,1002,502,2026-01-02,1800.0,completed,None
2,1003,501,2026-01-03,3200.0,completed,SAVE20
3,1004,503,2026-01-04,900.0,completed,None
4,1005,999,2026-01-05,700.0,completed,None
5,1006,504,2026-01-06,5000.0,cancelled,FEST
6,1007,505,2026-01-07,NaN,completed,None
7,1008,507,2026-01-08,1200.0,completed,A1


## 12. Selecting Specific Columns

Selecting only required columns is better for performance and readability.


In [ ]:
run_sql("""
SELECT
    order_id,
    customer_id,
    amount,
    status
FROM orders
""")


,order_id,customer_id,amount,status
0,1001,501,2500.0,completed
1,1002,502,1800.0,completed
2,1003,501,3200.0,completed
3,1004,503,900.0,completed
4,1005,999,700.0,completed
5,1006,504,5000.0,cancelled
6,1007,505,NaN,completed
7,1008,507,1200.0,completed


## 13. DISTINCT for De-duplication

`DISTINCT` returns unique values.


In [ ]:
run_sql("""
SELECT DISTINCT status
FROM orders
""")


,status
0,completed
1,cancelled


In [ ]:
run_sql("""
SELECT DISTINCT city
FROM customers
""")


,city
0,Delhi
1,Mumbai
2,Bangalore
3,Pune
4,Chennai
5,None


## 14. Column Alias using AS

Aliases make output column names more readable.


In [ ]:
run_sql("""
SELECT
    order_id AS order_number,
    amount AS order_amount,
    amount * 0.18 AS tax_amount,
    amount + (amount * 0.18) AS final_amount
FROM orders
WHERE amount IS NOT NULL
""")


,order_number,order_amount,tax_amount,final_amount
0,1001,2500.0,450.0,2950.0
1,1002,1800.0,324.0,2124.0
2,1003,3200.0,576.0,3776.0
3,1004,900.0,162.0,1062.0
4,1005,700.0,126.0,826.0
5,1006,5000.0,900.0,5900.0
6,1008,1200.0,216.0,1416.0


## 15. Table Alias using AS

Table aliases make joins easier to read.


In [ ]:
run_sql("""
SELECT
    o.order_id,
    c.customer_name,
    c.city,
    o.amount,
    o.status
FROM orders AS o
LEFT JOIN customers AS c
    ON o.customer_id = c.customer_id
""")


,order_id,customer_name,city,amount,status
0,1001,Riya,Delhi,2500.0,completed
1,1002,Aarav,Mumbai,1800.0,completed
2,1003,Riya,Delhi,3200.0,completed
3,1004,Kabir,Bangalore,900.0,completed
4,1005,None,None,700.0,completed
5,1006,Meera,Pune,5000.0,cancelled
6,1007,Anz,Chennai,NaN,completed
7,1008,Aliz,None,1200.0,completed


# Filtering Logic with WHERE


## 16. WHERE with Comparison Operators


In [ ]:
run_sql("""
SELECT *
FROM orders
WHERE amount > 1000
""")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1001,501,2026-01-01,2500.0,completed,NEW10
1,1002,502,2026-01-02,1800.0,completed,None
2,1003,501,2026-01-03,3200.0,completed,SAVE20
3,1006,504,2026-01-06,5000.0,cancelled,FEST
4,1008,507,2026-01-08,1200.0,completed,A1


## 17. WHERE with AND

Both conditions must be true.


In [ ]:
run_sql("""
SELECT *
FROM orders
WHERE amount > 1000
  AND status = 'completed'
""")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1001,501,2026-01-01,2500.0,completed,NEW10
1,1002,502,2026-01-02,1800.0,completed,None
2,1003,501,2026-01-03,3200.0,completed,SAVE20
3,1008,507,2026-01-08,1200.0,completed,A1


## 18. WHERE with OR

At least one condition must be true.


In [ ]:
run_sql("""
SELECT *
FROM orders
WHERE status = 'pending'
   OR status = 'cancelled'
""")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1006,504,2026-01-06,5000.0,cancelled,FEST


## 19. WHERE with NOT

`NOT` reverses a condition.


In [ ]:
run_sql("""
SELECT *
FROM orders
WHERE NOT status = 'completed'
""")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1006,504,2026-01-06,5000.0,cancelled,FEST


## 20. IN Operator

`IN` checks whether a value exists in a list of values.


In [ ]:
run_sql("""
SELECT *
FROM customers
WHERE city IN ('Delhi', 'Mumbai', 'Chennai')
""")


,customer_id,customer_name,city,segment,email
0,501,Riya,Delhi,Premium,riya@example.com
1,502,Aarav,Mumbai,Standard,aarav@example.com
2,505,Anz,Chennai,Standard,anz@example.com
3,506,AnushkaZ,Delhi,Premium,anushkaz@example.com


## 21. BETWEEN Operator

`BETWEEN` filters values inside a range.

In SQL, `BETWEEN` is inclusive.


In [ ]:
run_sql("""
SELECT *
FROM orders
WHERE amount BETWEEN 1000 AND 3000
""")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1001,501,2026-01-01,2500.0,completed,NEW10
1,1002,502,2026-01-02,1800.0,completed,None
2,1008,507,2026-01-08,1200.0,completed,A1


## 22. LIKE Operator with `%`

`%` means any number of characters.

Example:
- `A%` means starts with A
- `%z` means ends with z
- `%ar%` means contains ar


In [ ]:
run_sql("""
SELECT *
FROM customers
WHERE customer_name LIKE 'A%'
""")


,customer_id,customer_name,city,segment,email
0,502,Aarav,Mumbai,Standard,aarav@example.com
1,505,Anz,Chennai,Standard,anz@example.com
2,506,AnushkaZ,Delhi,Premium,anushkaz@example.com
3,507,Aliz,None,Standard,None


## 23. LIKE Operator with `_`

`_` means exactly one character.

Example:
- `A_rav` can match names where `_` is one character


In [ ]:
run_sql("""
SELECT *
FROM customers
WHERE customer_name LIKE 'A_rav'
""")


,customer_id,customer_name,city,segment,email
0,502,Aarav,Mumbai,Standard,aarav@example.com


## 24. Starts with A and Ends with Z

To find names that start with `A` and end with `Z`:


In [ ]:
run_sql("""
SELECT *
FROM customers
WHERE customer_name LIKE 'A%Z'
""")


,customer_id,customer_name,city,segment,email
0,505,Anz,Chennai,Standard,anz@example.com
1,506,AnushkaZ,Delhi,Premium,anushkaz@example.com
2,507,Aliz,None,Standard,None


## 25. IS NULL

Do not compare NULL using `= NULL`.

Use `IS NULL`.


In [ ]:
run_sql("""
SELECT *
FROM orders
WHERE coupon_code IS NULL
""")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1002,502,2026-01-02,1800.0,completed,None
1,1004,503,2026-01-04,900.0,completed,None
2,1005,999,2026-01-05,700.0,completed,None
3,1007,505,2026-01-07,NaN,completed,None


## 26. IS NOT NULL


In [ ]:
run_sql("""
SELECT *
FROM orders
WHERE coupon_code IS NOT NULL
""")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1001,501,2026-01-01,2500.0,completed,NEW10
1,1003,501,2026-01-03,3200.0,completed,SAVE20
2,1006,504,2026-01-06,5000.0,cancelled,FEST
3,1008,507,2026-01-08,1200.0,completed,A1


## 27. NULL Comparison Pitfall

The following query does not work as expected because NULL means unknown.

Use `IS NULL` instead.


In [ ]:
run_sql("""
SELECT *
FROM orders
WHERE coupon_code = NULL
""")


,order_id,customer_id,order_date,amount,status,coupon_code


# SELECT Pipeline and SQL Execution Order

Logical SQL execution order:

1. FROM
2. JOIN / ON
3. WHERE
4. GROUP BY
5. HAVING
6. SELECT
7. DISTINCT
8. ORDER BY
9. LIMIT

Even though `SELECT` is written first, the database logically starts from `FROM`.


## 28. Query Execution Order Example

The query is written as:

```sql
SELECT ...
FROM ...
WHERE ...
ORDER BY ...
LIMIT ...
```

But the database logically processes:

```text
FROM -> WHERE -> SELECT -> ORDER BY -> LIMIT
```


In [ ]:
run_sql("""
SELECT
    order_id,
    amount,
    status
FROM orders
WHERE amount > 1000
ORDER BY amount DESC
LIMIT 3
""")


,order_id,amount,status
0,1006,5000.0,cancelled
1,1003,3200.0,completed
2,1001,2500.0,completed


# Data Engineering Use Cases


## 29. Data Quality Check: Orders with Missing Amount


In [ ]:
run_sql("""
SELECT *
FROM orders
WHERE amount IS NULL
""")


,order_id,customer_id,order_date,amount,status,coupon_code
0,1007,505,2026-01-07,None,completed,None


## 30. Data Quality Check: Invalid Customer IDs

An invalid customer ID appears in `orders` but not in `customers`.


In [ ]:
run_sql("""
SELECT
    o.order_id,
    o.customer_id,
    o.amount,
    o.status
FROM orders AS o
LEFT JOIN customers AS c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
""")


,order_id,customer_id,amount,status
0,1005,999,700.0,completed


## 31. Clean Completed Orders

This query applies multiple filters to produce analytics-ready order records.


In [ ]:
run_sql("""
SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    c.city,
    o.order_date,
    o.amount,
    o.status,
    o.coupon_code
FROM orders AS o
LEFT JOIN customers AS c
    ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
  AND o.amount IS NOT NULL
  AND o.amount > 0
  AND c.customer_id IS NOT NULL
ORDER BY o.order_date
""")


,order_id,customer_id,customer_name,city,order_date,amount,status,coupon_code
0,1001,501,Riya,Delhi,2026-01-01,2500.0,completed,NEW10
1,1002,502,Aarav,Mumbai,2026-01-02,1800.0,completed,None
2,1003,501,Riya,Delhi,2026-01-03,3200.0,completed,SAVE20
3,1004,503,Kabir,Bangalore,2026-01-04,900.0,completed,None
4,1008,507,Aliz,None,2026-01-08,1200.0,completed,A1


# Practice Problems


## 32. Practice Problem 1

Insert one new customer into the `customers` table.

Then display all customers.


In [ ]:
# Write SQL here


## 33. Practice Problem 2

Insert three new orders using bulk insert.


In [ ]:
# Write Python + SQL here


## 34. Practice Problem 3

Select only `order_id`, `amount`, and `status` from the orders table.


In [ ]:
# Write SQL here


## 35. Practice Problem 4

Find all completed orders with amount between 1000 and 3000.


In [ ]:
# Write SQL here


## 36. Practice Problem 5

Find customers whose name starts with `A`.


In [ ]:
# Write SQL here


## 37. Practice Problem 6

Find customers where email is missing.


In [ ]:
# Write SQL here


## 38. Practice Problem 7

Update the city of customer_id 507 to `Hyderabad`.

Use a WHERE clause.


In [ ]:
# Write SQL here


## 39. Practice Problem 8

Delete orders where status is `cancelled`.

Use a demo table instead of deleting from the real `orders` table.


In [ ]:
# Write SQL here


## 40. Practice Problem 9

Find all orders where coupon code is not null and amount is greater than 1000.


In [ ]:
# Write SQL here


## 41. Practice Problem 10

Create a clean order extract with readable aliases:

Columns:
- order_number
- customer_name
- city
- order_amount
- order_status


In [ ]:
# Write SQL here


# Interview Questions


## 42. What is the execution order of a SQL query?

Logical execution order:

1. FROM
2. JOIN / ON
3. WHERE
4. GROUP BY
5. HAVING
6. SELECT
7. DISTINCT
8. ORDER BY
9. LIMIT

A basic SELECT query is often logically processed as:

```text
FROM -> WHERE -> SELECT -> ORDER BY -> LIMIT
```


## 43. How do you find all records where the name starts with A and ends with Z?

Use `LIKE` with `%` wildcard:

```sql
SELECT *
FROM customers
WHERE customer_name LIKE 'A%Z';
```

`%` means any number of characters.


## 44. How do you handle NULL values in a comparison?

Use:

```sql
IS NULL
```

or:

```sql
IS NOT NULL
```

Do not use:

```sql
= NULL
```

because NULL represents unknown, not a normal value.


## 45. Why is UPDATE or DELETE without WHERE dangerous?

Without `WHERE`:
- `UPDATE` modifies every row in the table
- `DELETE` removes every row in the table

Safe pattern:

```sql
UPDATE table_name
SET column_name = value
WHERE condition;
```

```sql
DELETE FROM table_name
WHERE condition;
```


## 46. Summary

Key takeaways:
- DML changes data using INSERT, UPDATE, and DELETE.
- Always use WHERE carefully with UPDATE and DELETE.
- SELECT retrieves data from tables.
- Avoid SELECT * in production queries when only specific columns are needed.
- DISTINCT removes duplicate output values.
- AS creates readable aliases.
- WHERE supports AND, OR, NOT, IN, BETWEEN, LIKE, IS NULL.
- NULL must be handled with IS NULL or IS NOT NULL.
- SQL is logically processed starting from FROM, not SELECT.
